**IMPORTS**

In [114]:
# Torch related imports
import torch
import torch.nn as nn

# Other libraries
import time

# Define the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

**UNet IMPLEMENTATION**

In [115]:
class DoubleConv(nn.Module):
    """
        Implementation of the double convolution
        module
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Defining the sequential module
        self.convolutions = nn.Sequential(
            # Two convolutions
            nn.Conv2d(in_channels, out_channels, kernel_size = 3, padding = 1, stride = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True),
            nn.Conv2d(out_channels, out_channels, kernel_size = 3, padding = 1, stride = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True)     
        )

    def forward(self, x:torch.Tensor):
        return self.convolutions(x)  

In [116]:
class DownSample(nn.Module):
    """ 
        Implementing downsampling blocks
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Getting the double convolution
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

    def forward(self, x:torch.Tensor):
        conv_out = self.conv(x)
        down = self.pool(conv_out)
        return conv_out, down

In [ ]:
class UpSample(nn.Module):
    """  
        Implementing upsampling block
    
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Define the upsample
        self.ups = nn.ConvTranspose2d(in_channels, in_channels//2, kernel_size = 2, stride = 2)

        # Define the double conv
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1:torch.Tensor, x2:torch.Tensor):

        # Upsampling
        x1 = self.ups(x1)

        # Concat process --> concat channels 256 + 256
        x = torch.cat([x1,x2],1)

        return self.conv(x)

In [118]:
class UNet(nn.Module):
    """ 
        Implementation of UNet general architecture
    """
    def __init__(self, in_channels, num_classes):
        super().__init__()

        # Define the downsampling part
        self.down_convolution_1 = DownSample(in_channels, 64)
        self.down_convolution_2 = DownSample(64, 128)
        self.down_convolution_3 = DownSample(128, 256)
        self.down_convolution_4 = DownSample(256, 512)

        # Define the bottleneck
        self.bottleneck =  DoubleConv(512, 1024)

        # Define the upsampling part
        self.up_convolution_1 = UpSample(1024, 512)
        self.up_convolution_2 = UpSample(512, 256)
        self.up_convolution_3 = UpSample(256, 128)
        self.up_convolution_4 = UpSample(128, 64)

        # Defining the output conv
        self.out_conv = nn.Conv2d(in_channels = 64, out_channels = num_classes, kernel_size = 1)

    def forward(self, x:torch.Tensor):

        # Downsampling
        conv_out_1, down_1 = self.down_convolution_1(x)
        conv_out_2, down_2 = self.down_convolution_2(down_1)
        conv_out_3, down_3 = self.down_convolution_3(down_2)
        conv_out_4, down_4 = self.down_convolution_4(down_3)

        # The bottleneck
        b = self.bottleneck(down_4)

        #Upsampling
        up_1  = self.up_convolution_1(b, conv_out_4)
        up_2  = self.up_convolution_2(up_1, conv_out_3)
        up_3  = self.up_convolution_3(up_2, conv_out_2)
        up_4  = self.up_convolution_4(up_3, conv_out_1)

        # Defining the outout
        out = self.out_conv(up_4)

        return out

**TESTING THE IMPLEMENTATION**

In [119]:
def fast_test(device = "cpu"):
    """
        Function aimed to time profile
    """

    # Start time
    start = time.time()
    print('*'*50)
    print(f'         LITTLE EXAMPLE PROFILING ({device})            ')
    print('*'*50)

    # Plotting parameters
    print(f"Input Channels ({3}) -- Output Channels({10})")
    #  Create test image
    image = torch.rand((1, 3, 512, 512)).to(device)
    print('·Original Shape:',image.shape)

    # Create the model
    in_channels = 3
    out_channels = 10
    model = UNet(in_channels, out_channels)

    # Move model to the selected device
    model.to(device)

    # Test the model
    output = model(image)
    print('·Output Shape:',output.shape)

    # Elapsed time
    print(f'·Elapsed Time --> {time.time()-start}s')
    print()


In [120]:
# Calling the test function 
fast_test(device)
fast_test('cpu')

**************************************************
         LITTLE EXAMPLE PROFILING (cuda)            
**************************************************
Input Channels (3) -- Output Channels(10)
·Original Shape: torch.Size([1, 3, 512, 512])
·Output Shape: torch.Size([1, 10, 512, 512])
·Elapsed Time --> 0.7527482509613037s

**************************************************
         LITTLE EXAMPLE PROFILING (cpu)            
**************************************************
Input Channels (3) -- Output Channels(10)
·Original Shape: torch.Size([1, 3, 512, 512])
·Output Shape: torch.Size([1, 10, 512, 512])
·Elapsed Time --> 2.203171730041504s



**MODEL'S DIAGRAM**

1. **Input_Channels** = 3
2. **Ouput_Channels** = 10

In [121]:
# Define the parameters
in_c = 3
out_ch = 10
UNet(3,10)

UNet(
  (down_convolution_1): DownSample(
    (conv): DoubleConv(
      (convolutions): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (down_convolution_2): DownSample(
    (conv): DoubleConv(
      (convolutions): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
       